In [ ]:
import sys
from importlib import reload
from pathlib import Path

import torch
from sklearn.metrics import balanced_accuracy_score

sys.path.append("..")
from src import plotly_plots as pp
from src import tagger_model, util
from src import torch_metrics as tm
from src import torch_util as tu


In [ ]:
reload(tu)
conf = tagger_model.RNNTaggerConfig(
    mlp_sizes=[80],
)

model = tagger_model.RNNTagger(
    conf,
    vocab_sz_token=10,
    vocab_sz_tag=7,
)

tokens = torch.tensor([[1, 2, 3], [3, 4, 0]])
tags_det = torch.tensor([[1, 1, 5], [3, 2, 0]])

out = model(tokens, tags_det)
print(model.tot_weights)
out.shape


In [ ]:
conf = tagger_model.RNNTaggerConfig(
    d_emb_token=2,
    d_emb_tag=2,
    d_hidden_rnn=2,
    rnn_variant="rnn",
    n_rnn_layers=1,
    mlp_sizes=[7],
)

model = tagger_model.RNNTagger(
    conf,
    vocab_sz_token=4,
    vocab_sz_tag=4,
)
print(f"{model.tot_weights = }")
pars = list(model.named_parameters())
for name, p in pars:
    print(name, p.shape)

## Data


In [ ]:
# load data, convert to dataframe
reload(tu)
reload(util)
reload(tagger_model)

split_idx, split_date = util.load_split_idx()
print(f"Loaded split {split_date}")
data = {
    sk: util.dataset_to_df(v)
    for sk, v in util.load_dataset_splits(split_idx, path=Path("../data/dataset.ndjson")).items()
}
# get a vocab
vocab, token2idx, tag_vocab, tag2idx = util.make_vocab(data["train"])

device = tu.get_dev()

print(f"\n{len(vocab)=} | {len(tag_vocab)=} | {device=}\n")

dsets = {
    k: tu.SequenceDataset.from_dataframe(df, token2idx, tag2idx, device="cpu")
    for k, df in data.items()
}
for k, d in dsets.items():
    print(f"{k}: {d}")

## class weight?


In [ ]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

cls_sorted = data["train"]["tags"].explode().unique().sort()
clsk = compute_class_weight(
    "balanced",
    classes=np.array([tag2idx[t] for t in cls_sorted]),
    y=np.array([tag2idx[t] for t in data["train"]["tags"].explode()]),
)

class_weights = dict(zip(cls_sorted, clsk))
clw_sk = torch.tensor(
    [class_weights.get(tag, 1) for tag in tag_vocab],
    dtype=torch.float32,
).to(device)
# for tag, w in zip(tag_vocab, clw_sk):
#     print(f"{tag:8}  {w:.3f}")

## training


In [ ]:
reload(tagger_model)
conf = tagger_model.RNNTaggerConfig(
    d_emb_token=16,
    d_emb_tag=16,
    d_hidden_rnn=16,
    rnn_variant="rnn",
    n_rnn_layers=1,
    mlp_sizes=[64],
    bidi=True,
    dropout_rnn=0.0,
)

model = tagger_model.RNNTagger(
    conf,
    vocab_sz_token=len(vocab),
    vocab_sz_tag=len(tag_vocab),
)
model.to(device=device)


train_settings = tagger_model.TrainSettings(
    label_smoothing=0.1,
    weight_decay=0.1,
    start_lr=5e-3,
    bs_train=16,
    loss_weights=None,
    plateau_lr_patience=20,
    stop_patience=25,
    max_epochs=20,
)
metrics = model.complete_train_loop(train_settings, dsets["train"], dsets["val"], verbose=True)
print(metrics.keys())
pp.train_metrics_single_run(metrics)

## metrics


In [ ]:
yt = torch.tensor([0, 0, 0, 1])
yp = torch.tensor([0, 0, 1, 1])

acc = (yt == yp).mean(dtype=torch.float32)


print(f"{acc=}")
print(f"{tm.balanced_acc(yp, yt)=}")
print(f"{balanced_accuracy_score(yt,yp,adjusted=False)=}")


In [ ]:
print(yt.reshape(-1, 1) == yt)
print((yt.reshape(-1, 1) == yt).sum(0))